In [1]:
import os
import json
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix


In [2]:
MODEL_DIR = r"../model/final_model"
BASE_MODEL_NAME = "microsoft/graphcodebert-base"

MAX_LENGTH = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)
print("MODEL_DIR exists =", os.path.exists(MODEL_DIR))
print("MODEL_DIR abs =", os.path.abspath(MODEL_DIR))


DEVICE = cpu
MODEL_DIR exists = True
MODEL_DIR abs = c:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-java\model\final_model


In [3]:
tokenizer_files = [
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "special_tokens_map.json",
]

has_local_tokenizer = any(os.path.exists(os.path.join(MODEL_DIR, f)) for f in tokenizer_files)

if has_local_tokenizer:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    print("Loaded tokenizer từ MODEL_DIR")
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    print("MODEL_DIR không có tokenizer files -> dùng tokenizer gốc:", BASE_MODEL_NAME)


MODEL_DIR không có tokenizer files -> dùng tokenizer gốc: microsoft/graphcodebert-base


In [4]:

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

print(model.__class__.__name__)
print("num_labels =", model.config.num_labels)
print("id2label  =", getattr(model.config, "id2label", None))
print("label2id  =", getattr(model.config, "label2id", None))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification
num_labels = 2
id2label  = {0: 'LABEL_0', 1: 'LABEL_1'}
label2id  = {'LABEL_0': 0, 'LABEL_1': 1}


## Quan trọng về label

Với bài toán binary classification, thường có 2 khả năng:

- `0 = non-vulnerable`, `1 = vulnerable`
- hoặc ngược lại

Nếu lúc train bạn **không set `id2label/label2id` rõ ràng**, thì cần tự chỉ định mapping ở cell dưới.

Mặc định notebook này dùng:
- `0 -> non-vulnerable`
- `1 -> vulnerable`

Nếu predict ra bị ngược logic, hãy đổi lại `ID2LABEL`.


In [5]:

if getattr(model.config, "id2label", None) and len(model.config.id2label) == model.config.num_labels:
    # chuẩn hoá key về int nếu cần
    ID2LABEL = {}
    for k, v in model.config.id2label.items():
        try:
            ID2LABEL[int(k)] = v
        except:
            ID2LABEL[k] = v
else:
    ID2LABEL = {
        0: "non-vulnerable",
        1: "vulnerable"
    }

LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print("ID2LABEL =", ID2LABEL)


ID2LABEL = {0: 'LABEL_0', 1: 'LABEL_1'}


In [6]:
def predict_one(code: str, max_length: int = MAX_LENGTH):
    inputs = tokenizer(
        code,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().tolist()
        pred_id = int(torch.argmax(logits, dim=-1).item())

    pred_label = ID2LABEL.get(pred_id, str(pred_id))

    result = {
        "pred_id": pred_id,
        "pred_label": pred_label,
        "probabilities": {ID2LABEL.get(i, str(i)): float(p) for i, p in enumerate(probs)}
    }
    return result


In [7]:

sample_code = r'''
public boolean login(String username, String password, Connection conn) throws Exception {
    String sql = "SELECT * FROM users WHERE username = '" + username + "' AND password = '" + password + "'";
    Statement st = conn.createStatement();
    ResultSet rs = st.executeQuery(sql);
    return rs.next();
}
'''

result = predict_one(sample_code)
print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "pred_id": 1,
  "pred_label": "LABEL_1",
  "probabilities": {
    "LABEL_0": 0.4384685754776001,
    "LABEL_1": 0.5615314245223999
  }
}


In [8]:

def predict_batch(codes, batch_size: int = 8, max_length: int = MAX_LENGTH):
    all_rows = []

    for i in range(0, len(codes), batch_size):
        batch_codes = codes[i:i+batch_size]

        inputs = tokenizer(
            batch_codes,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).detach().cpu()
            pred_ids = torch.argmax(logits, dim=-1).detach().cpu().tolist()

        for code, pred_id, prob_vec in zip(batch_codes, pred_ids, probs):
            row = {
                "code": code,
                "pred_id": int(pred_id),
                "pred_label": ID2LABEL.get(int(pred_id), str(pred_id)),
            }
            for label_id, p in enumerate(prob_vec.tolist()):
                row[f"prob_{ID2LABEL.get(label_id, label_id)}"] = float(p)
            all_rows.append(row)

    return pd.DataFrame(all_rows)


In [9]:
sample_tests = [
    {
        "id": 1,
        "description": "SQL Injection - vulnerable",
        "expected_label": "vulnerable",
        "code": '''public boolean login(String username, String password, Connection conn) throws Exception {
    String sql = "SELECT * FROM users WHERE username = '" + username + "' AND password = '" + password + "'";
    Statement st = conn.createStatement();
    ResultSet rs = st.executeQuery(sql);
    return rs.next();
}'''
    },
    {
        "id": 2,
        "description": "SQL Injection - safe with PreparedStatement",
        "expected_label": "non-vulnerable",
        "code": '''public boolean login(String username, String password, Connection conn) throws Exception {
    String sql = "SELECT * FROM users WHERE username = ? AND password = ?";
    PreparedStatement ps = conn.prepareStatement(sql);
    ps.setString(1, username);
    ps.setString(2, password);
    ResultSet rs = ps.executeQuery();
    return rs.next();
}'''
    },
    {
        "id": 3,
        "description": "Path traversal - vulnerable",
        "expected_label": "vulnerable",
        "code": '''public String readFile(String name) throws Exception {
    File file = new File("/app/data/" + name);
    BufferedReader br = new BufferedReader(new FileReader(file));
    return br.readLine();
}'''
    },
    {
        "id": 4,
        "description": "Path traversal - safe normalize path",
        "expected_label": "non-vulnerable",
        "code": '''public String readFile(String name) throws Exception {
    Path base = Paths.get("/app/data").toRealPath();
    Path target = base.resolve(name).normalize();
    if (!target.startsWith(base)) {
        throw new SecurityException("invalid path");
    }
    return Files.readString(target);
}'''
    },
    {
        "id": 5,
        "description": "Command injection - vulnerable",
        "expected_label": "vulnerable",
        "code": '''public void ping(String host) throws Exception {
    Runtime.getRuntime().exec("ping " + host);
}'''
    },
    {
        "id": 6,
        "description": "Command execution - safer fixed command shape",
        "expected_label": "non-vulnerable",
        "code": '''public void ping() throws Exception {
    new ProcessBuilder("ping", "127.0.0.1").start();
}'''
    },
    {
        "id": 7,
        "description": "Hard-coded password",
        "expected_label": "vulnerable",
        "code": '''public Connection getConn() throws Exception {
    String password = "admin123";
    return DriverManager.getConnection("jdbc:mysql://localhost/test", "root", password);
}'''
    },
    {
        "id": 8,
        "description": "Password from environment",
        "expected_label": "non-vulnerable",
        "code": '''public Connection getConn() throws Exception {
    String password = System.getenv("DB_PASSWORD");
    return DriverManager.getConnection("jdbc:mysql://localhost/test", "root", password);
}'''
    }
]

sample_df = pd.DataFrame(sample_tests)
sample_df


,id,description,expected_label,code
0,1,SQL Injection - vulnerable,vulnerable,"public boolean login(String username, String p..."
1,2,SQL Injection - safe with PreparedStatement,non-vulnerable,"public boolean login(String username, String p..."
2,3,Path traversal - vulnerable,vulnerable,public String readFile(String name) throws Exc...
3,4,Path traversal - safe normalize path,non-vulnerable,public String readFile(String name) throws Exc...
4,5,Command injection - vulnerable,vulnerable,public void ping(String host) throws Exception...
5,6,Command execution - safer fixed command shape,non-vulnerable,public void ping() throws Exception {\n new...
6,7,Hard-coded password,vulnerable,public Connection getConn() throws Exception {...
7,8,Password from environment,non-vulnerable,public Connection getConn() throws Exception {...


In [10]:

pred_df = predict_batch(sample_df["code"].tolist())

result_df = sample_df.copy()
result_df["pred_id"] = pred_df["pred_id"]
result_df["pred_label"] = pred_df["pred_label"]

# thêm xác suất
prob_cols = [c for c in pred_df.columns if c.startswith("prob_")]
for c in prob_cols:
    result_df[c] = pred_df[c]

result_df[["id", "description", "expected_label", "pred_label"] + prob_cols]


,id,description,expected_label,pred_label,prob_LABEL_0,prob_LABEL_1
0,1,SQL Injection - vulnerable,vulnerable,LABEL_1,0.438413,0.561587
1,2,SQL Injection - safe with PreparedStatement,non-vulnerable,LABEL_1,0.428936,0.571064
2,3,Path traversal - vulnerable,vulnerable,LABEL_1,0.456621,0.543379
3,4,Path traversal - safe normalize path,non-vulnerable,LABEL_1,0.471899,0.528101
4,5,Command injection - vulnerable,vulnerable,LABEL_1,0.448931,0.551069
5,6,Command execution - safer fixed command shape,non-vulnerable,LABEL_1,0.455088,0.544912
6,7,Hard-coded password,vulnerable,LABEL_1,0.458185,0.541815
7,8,Password from environment,non-vulnerable,LABEL_1,0.461643,0.538357


In [11]:
# Cell 12 - Tính accuracy trên sample test tự tạo
y_true = result_df["expected_label"].tolist()
y_pred = result_df["pred_label"].tolist()

acc = accuracy_score(y_true, y_pred)
print("Sample accuracy =", round(acc, 4))
print()
print(classification_report(y_true, y_pred, digits=4))


Sample accuracy = 0.0

                precision    recall  f1-score   support

       LABEL_1     0.0000    0.0000    0.0000       0.0
non-vulnerable     0.0000    0.0000    0.0000       4.0
    vulnerable     0.0000    0.0000    0.0000       4.0

      accuracy                         0.0000       8.0
     macro avg     0.0000    0.0000    0.0000       8.0
  weighted avg     0.0000    0.0000    0.0000       8.0



c:\Users\Admin\Documents\miniconda\envs\nckh\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Documents\miniconda\envs\nckh\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Documents\miniconda\envs\nckh\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

In [ ]:
def find_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def evaluate_csv(csv_path, save_pred_path=None):
    df = pd.read_csv(csv_path)

    code_col = find_first_existing_column(df, ["code", "func", "function", "source", "input"])
    label_col = find_first_existing_column(df, ["label", "target", "labels", "y"])

    if code_col is None:
        raise ValueError(f"Không tìm thấy cột code trong file. Các cột hiện có: {list(df.columns)}")

    pred_df = predict_batch(df[code_col].astype(str).tolist())

    out_df = df.copy()
    out_df["pred_id"] = pred_df["pred_id"]
    out_df["pred_label"] = pred_df["pred_label"]

    prob_cols = [c for c in pred_df.columns if c.startswith("prob_")]
    for c in prob_cols:
        out_df[c] = pred_df[c]

    metrics = None
    if label_col is not None:
        y_true_raw = out_df[label_col].tolist()

        # Chuẩn hoá label thật về cùng dạng string với pred_label
        normalized_true = []
        for x in y_true_raw:
            if isinstance(x, str):
                normalized_true.append(x.strip())
            else:
                # giả sử label dạng số 0/1
                x = int(x)
                normalized_true.append(ID2LABEL.get(x, str(x)))

        y_true = normalized_true
        y_pred = out_df["pred_label"].tolist()

        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label="vulnerable", zero_division=0)
        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": p,
            "recall": r,
            "f1": f1,
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["non-vulnerable", "vulnerable"]).tolist(),
            "classification_report": classification_report(y_true, y_pred, digits=4)
        }

    if save_pred_path is not None:
        out_df.to_csv(save_pred_path, index=False)
        print("Đã lưu file prediction vào:", save_pred_path)

    return out_df, metrics


In [16]:
TEST_CSV_PATH = r"../data/sample_java_vul_test.csv"

if os.path.exists(TEST_CSV_PATH):
    out_df, metrics = evaluate_csv(TEST_CSV_PATH, save_pred_path="./test_predictions.csv")
    display(out_df.head())

    if metrics is not None:
        print(json.dumps(
            {k: v for k, v in metrics.items() if k != "classification_report"},
            indent=2,
            ensure_ascii=False
        ))
        print()
        print(metrics["classification_report"])
else:
    print("Chưa tìm thấy file:", TEST_CSV_PATH)


Đã lưu file prediction vào: ./test_predictions.csv


,id,description,expected_label,code,pred_id,pred_label,prob_LABEL_0,prob_LABEL_1
0,1,SQL Injection - vulnerable,vulnerable,"public boolean login(String username, String p...",1,LABEL_1,0.438413,0.561587
1,2,SQL Injection - safe with PreparedStatement,non-vulnerable,"public boolean login(String username, String p...",1,LABEL_1,0.428936,0.571064
2,3,Path traversal - vulnerable,vulnerable,public String readFile(String name) throws Exc...,1,LABEL_1,0.456621,0.543379
3,4,Path traversal - safe normalize path,non-vulnerable,public String readFile(String name) throws Exc...,1,LABEL_1,0.471899,0.528101
4,5,Command injection - vulnerable,vulnerable,public void ping(String host) throws Exception...,1,LABEL_1,0.448931,0.551069


In [14]:


custom_code = r'''
public String unsafeRead(String fileName) throws Exception {
    return Files.readString(Paths.get("/tmp/" + fileName));
}
'''

predict_one(custom_code)


{'pred_id': 1,
 'pred_label': 'LABEL_1',
 'probabilities': {'LABEL_0': 0.4685339629650116,
  'LABEL_1': 0.5314660668373108}}

In [15]:
# Cell 16 - Một sanity check để xác nhận mapping 0/1 có bị ngược hay không
very_safe = r'''
public int add(int a, int b) {
    return a + b;
}
'''

very_risky = r'''
public void run(String cmd) throws Exception {
    Runtime.getRuntime().exec(cmd);
}
'''

print("SAFE SAMPLE:")
print(json.dumps(predict_one(very_safe), indent=2, ensure_ascii=False))
print()
print("RISKY SAMPLE:")
print(json.dumps(predict_one(very_risky), indent=2, ensure_ascii=False))


SAFE SAMPLE:
{
  "pred_id": 1,
  "pred_label": "LABEL_1",
  "probabilities": {
    "LABEL_0": 0.44701075553894043,
    "LABEL_1": 0.5529891848564148
  }
}

RISKY SAMPLE:
{
  "pred_id": 1,
  "pred_label": "LABEL_1",
  "probabilities": {
    "LABEL_0": 0.4407317638397217,
    "LABEL_1": 0.5592682361602783
  }
}
